# **Deep Learning and Neural Networks**
# **Week 9 Lab: AI Reasoning with LLMs**

This lab explores techniques for solving reasoning problems with large language models (LLMs). It consists of three tasks:
- **Task 1**: Use Zero-Shot Chain-of-Thought (CoT) prompting to solve a reasoning problem. This task includes two questions.
- **Task 2**: Use Self-Consistency (SC) prompting to solve a reasoning problem. This task includes two questions.
- **Task 3**: Implement a simple reasoning task using an LLM pre-trained for reasoning. This task includes two questions.

**Note for all tasks:** Depending on the problem, one reasoning technique may produce a more accurate result than another. This is especially true when we use small models, like we are using in this lab. Therefore, rather than focusing only on whether the final answer is correct, pay close attention to the reasoning process the model follows and identify where or how it may fail.

**Important:** Using a free GPU in Google Colab will speed up execution for all tasks in this lab. To enable a GPU, go to "Runtime", select "Change runtime type", choose one of the available GPUs under "Hardware accelerator" (for example, T4 GPU), and click "Save".

## **General Instructions**
1. Read the explanatory text and run each code cell in sequence.
2. Questions for each task appear throughout the notebook. Add your answers by following the specific instructions. Some questions require code, while others require analysis based on one or more outputs.
3. After completing all questions, submit your notebook through the corresponding QMPlus link for this lab assignment.

First, let's import the required libraries.

In [1]:
from datasets import load_dataset #To load the GSM8K dataset, which is a collection of grade school math problems, for testing the model's reasoning capabilities.
import re #Regular expressions module for pattern matching and text manipulation.
from collections import Counter #Counter is a subclass of dict that helps count hashable objects. It is used to count the occurrences of elements in an iterable, such as a list or string, and returns a dictionary-like object where keys are the elements and values are their counts.

import torch #PyTorch library for tensor computations and deep learning.
from transformers import AutoModelForCausalLM, AutoTokenizer #Transformers library from Hugging Face, which provides pre-trained models and tokenizers for natural language processing tasks. AutoModelForCausalLM is used for causal language modeling, and AutoTokenizer is used to convert text into tokens that the model can understand.

In [2]:
# Load dataset gsm8k
ds = load_dataset("openai/gsm8k", "main")

random_index = 0
sample = ds["test"][random_index] #and select a random question for our experiments

question = sample["question"]
answer = sample["answer"].split("####")
answer_path = answer[0]
final_answer = answer[1].strip()

print("Selected index:", random_index)
print("Question:", question)
print("Answer Path:", answer_path)
print("Final Answer:", final_answer)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Selected index: 0
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Answer Path: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.

Final Answer: 18


First, let's load a small model: Qwen2.5-0.5B-Instruct. We will use this model for Tasks 1 and 2.

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Using device: cuda


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## Task 1: Zero-Shot Chain-of-Thought Prompting

In this task, we will answer the selected question below using Chain-of-Thought (CoT) prompting. CoT is an inference-time strategy that encourages the model to reason step by step before producing a final answer. In the original CoT approach, exemplars are provided to guide the model through this process. The original CoT paper is available here: https://proceedings.neurips.cc/paper/2022/hash/9d5609613524ecf4f15af0f7b31abca4-Abstract-Conference.html. Later, zero-shot CoT was proposed as an alternative. The zero-shot CoT paper is available here: https://proceedings.neurips.cc/paper_files/paper/2022/hash/8bb0d291acd4acf06ef112099c16f326-Abstract-Conference.html.

In [4]:
#We will define a function for prompting our model using CoT (Chain of Thought) reasoning. The model will be instructed to first reason step-by-step and then provide the final answer.
def generate_cot_zeroshot(prompt: str, max_new_tokens: int = 250):
    messages = [
        {"role": "system", "content": "Solve the problem. First, write down your step-by-step reasoning. Finally, write the final answer clearly marked as 'Answer: <value>'."}, #Instructions for the model to reason step-by-step and provide the final answer
        {"role": "user", "content": prompt} #Our question is provided as a user message to the model
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) #Apply the chat template to format the messages for the model

    inputs = tokenizer(text, return_tensors="pt").to(device) #Tokenize the formatted text and move it to the appropriate device (GPU or CPU)

    with torch.no_grad(): #Disable gradient calculation for inference to save memory and computation
        outputs = model.generate(
            **inputs, #Pass the tokenized inputs to the model for generation
            max_new_tokens=max_new_tokens, #Limit the number of new tokens generated by the model
            do_sample=False, #Apply greedy decoding as in CoT
        )

    input_length = inputs.input_ids.shape[1] #Get the length of the input tokens to slice the output and retrieve only the generated part
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True) #Decode the generated tokens into a string, skipping any special tokens used by the model

print("CoT Answer:\n", generate_cot_zeroshot(question, max_new_tokens=500)) #Print the CoT answer

CoT Answer:
 To determine how much Janet makes at the farmers' market each day, we need to follow these steps:

1. Calculate the total number of eggs laid by the ducks in a day.
2. Subtract the number of eggs eaten in the morning from the total number of eggs laid.
3. Subtract the number of eggs baked from the remaining eggs after breakfast.
4. Determine how many ducks are laid in a day (since each duck lays one egg).
5. Calculate the revenue from selling the eggs.

Let's go through these steps in detail:

1. **Calculate the total number of eggs laid by the ducks in a day:**
   - Breakfast: 3 eggs
   - Baking: 4 eggs
   - Total eggs laid = 3 + 4 = 7 eggs

2. **Subtract the number of eggs eaten in the morning from the total number of eggs laid:**
   - Eggs eaten in the morning = 3
   - Remaining eggs after breakfast = 7 - 3 = 4 eggs

3. **Subtract the number of eggs baked from the remaining eggs after breakfast:**
   - Eggs baked = 4
   - Remaining eggs after baking = 4 - 4 = 0 eggs

4.

**Task 1 - Question 1:** What is the benefit of using an inference-time prompting technique such as Chain-of-Thought prompting, rather than fine-tuning a pre-trained model on labelled data for the specific task?

TODO - The main advantage of Chain-of-Thought prompting is that it does not require additional model training. Instead, the prompt encourages an already-trained model to work through a problem step by step before producing its final answer. This makes the approach quick and inexpensive to apply. There is no need to build a training loop, update the model’s weights or collect a large labelled dataset. It also avoids the risk of the model overfitting to a small fine-tuning dataset because no additional training takes place.

Fine-tuning, by comparison, requires labelled task-specific data, computing resources and time to train the model. It also changes the model’s weights so that it becomes more specialised for a particular task.

Chain-of-Thought prompting is more flexible because the same general-purpose model can be used for different reasoning tasks simply by changing the prompt. It is therefore useful when labelled data are limited or expensive to obtain. However, it does not guarantee correct reasoning, and its performance may depend strongly on how clearly the prompt is written.

**Task 1 - Question 2:** What is the benefit of using zero-shot Chain-of-Thought prompting instead of the original Chain-of-Thought prompting approach (that is, providing a few exemplars of the multi-step reasoning process)?

TODO - The benefit of zero shot Chain of Thought include:
- No example engineering required: Eliminates the need to carefully craft exemplars that demonstrate the correct reasoning pattern, which can be time-consuming and may require domain expertise.

- General applicability: Works for any problem without needing to find or create relevant examples, making it more universally applicable.

- Fewer tokens consumed: Since no exemplars are provided, fewer tokens are used in the prompt, reducing computational cost and avoiding context length limitations.

- Avoids exemplar bias: Few-shot examples can introduce bias or constrain the model's thinking to follow patterns that may not be optimal. Zero-shot allows more flexibility.

- Simplicity: The "Let's think step by step" approach is simpler to implement and requires no prior knowledge of what constitutes a good example.

- Works for novel problems: Particularly valuable when solving entirely new types of problems where appropriate exemplars don't exist.

## Task 2: Self-Consistency Prompting

Self-consistency prompting is another inference-time technique that guides a model to generate answers through a multi-step reasoning process. The paper is available here: https://arxiv.org/abs/2203.11171.

Although self-consistency prompting can also use exemplars to guide generation, in the next cell we will avoid using exemplars; that is, we will use zero-shot prompting.

In [5]:
#We will build a function to wrap the model generation in a self-consistency framework. The model will be prompted multiple times to generate different reasoning paths, and we will collect the final answers from each path. The most common answer will be returned as the consensus answer.
def generate_self_consistency(prompt: str, num_paths: int = 5) -> str:
    messages = [
        {"role": "system", "content": "Solve the problem. Show your work step-by-step. End your response with 'Answer: <number>'."}, #Instructions for the model to reason step-by-step and provide the final answer
        {"role": "user", "content": prompt} #Our question is provided as a user message to the model
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) #Apply the chat template to format the messages for the model
    inputs = tokenizer(text, return_tensors="pt").to(device) #Tokenize the formatted text and move it to the appropriate device (GPU or CPU)

    answers = []

    batch_inputs = {k: v.repeat(num_paths, 1) for k, v in inputs.items()} #Batch the generations for parallel processing. We repeat the inputs 'num_paths' times

    with torch.no_grad(): #Disable gradient calculation for inference to save memory and computation
        outputs = model.generate(
            **batch_inputs, #Pass the tokenized inputs to the model for generation
            max_new_tokens=500, #Limit the number of new tokens generated by the model
            do_sample=True,        # Enable sampling
            temperature=0.7,       # Introduce variance
        )

    input_length = inputs.input_ids.shape[1] #Get the length of the input tokens to slice the output and retrieve only the generated part

    print("--- Sampled Paths ---")
    for i, seq in enumerate(outputs): #Iterate through each generated sequence
        decoded = tokenizer.decode(seq[input_length:], skip_special_tokens=True) #Decode the generated tokens into a string, skipping any special tokens used by the model
        print(f"Path {i+1}:\n{decoded}\n{'-'*30}") #Print the generated reasoning path for each sample

        # Simple extraction logic for the final numeric answer (looking for "Answer: X" or a trailing digit)
        match = re.search(r"Answer:\s*([\d\.]+)", decoded, re.IGNORECASE)
        if match:
            answers.append(match.group(1))
        else:
            # Fallback to the last sequence of digits found
            digits = re.findall(r"\d+", decoded)
            if digits:
                answers.append(digits[-1])

    #Majority Vote
    if not answers:
        return "No valid numeric answers found."

    vote_counter = Counter(answers) #Count the occurrences of each answer to determine the most common one
    most_common_answer, count = vote_counter.most_common(1)[0] #Get the most common answer and its count

    print(f"Votes collected: {dict(vote_counter)}") #Print the collected votes for transparency
    return f"Consensus Answer: {most_common_answer} (with {count}/{num_paths} votes)" #Return the most common answer along with the number of votes it received out of the total number of paths generated

print(generate_self_consistency(question, num_paths=5)) #Print the consensus answer obtained from the self-consistency framework

--- Sampled Paths ---
Path 1:
To solve this, we need to first calculate how many eggs Janet's ducks lay each day, then subtract the number of eggs she eats in a day, and finally, subtract the cost of baking muffins from the remaining eggs.

Step 1: Calculate total eggs laid by ducks each day.
Since there are 16 eggs laid per day by ducks and Janet eats 3 eggs for breakfast, she would have:
\[ 16 - 3 = 13 \text{ eggs} \]

Step 2: Subtract the number of eggs eaten in a day.
After eating 3 eggs in the morning, she has:
\[ 13 - 3 = 10 \text{ eggs left} \]

Step 3: Subtract the cost of baking muffins.
The muffins Janet bakes cost her $4, so she loses that amount in a day:
\[ 10 - 4 = 6 \text{ eggs left} \]

Step 4: Calculate earnings from selling the remaining eggs.
She sells each egg for $2, so the total earnings are:
\[ 6 \times 2 = 12 \text{ dollars} \]

Therefore, Janet makes $12 every day at the farmers' market. The final answer is:
Answer: $12.
------------------------------
Path 2:
T

**Task 2 - Question 1:** What is the benefit of self-consistency prompting, as opposed to Chain-of-Thought (CoT) prompting, for reasoning problems?

TODO - Self-consistency prompting improves reasoning accuracy over standard
Chain-of-Thought (CoT) by sampling multiple diverse reasoning paths and using
a majority vote to pick the most frequent final answer, rather than relying on
a single greedy-decoded path.

Core benefits:

1. Improved accuracy through aggregation: plain CoT generates one reasoning
   path using greedy decoding, so if the model makes a mistake anywhere along
   that path, the final answer is simply wrong, with no way to catch it.
   Self-consistency samples several independent paths instead and takes a
   majority vote, so a mistake made in only one or two paths can be outvoted
   by paths that reach the correct answer through different valid routes.

2. Robustness to random errors: LLMs can make arbitrary slips or lucky
   guesses on any single attempt. Sampling several paths and aggregating them
   smooths out this randomness rather than trusting one roll of the dice.

3. Handling diverse reasoning routes: complex problems can often be solved
   through more than one valid line of reasoning. Self-consistency captures
   this by letting the model explore different approaches to the same
   problem, similar to how a person might double-check a difficult
   calculation using a different method.

4. No additional training required: like CoT itself, this benefit comes
   purely from how the model is prompted and sampled at inference time. The
   original self-consistency paper reports substantial accuracy gains on
   benchmarks such as GSM8K using this technique, with no fine-tuning needed.

**Task 2 - Question 2:** What is the role of the temperature parameter in self-consistency prompting? **Note:** You can experiment by changing this parameter in the code cell above to observe what happens.

TODO - Temperature controls the randomness and diversity of the sampling used to generate each reasoning path. Self consistency relies on producing different chains that it can then vote over, so temperature is what creates that diversity.

When the temperature is very low and close to zero, generation becomes almost deterministic and behaves like greedy decoding. All of the sampled paths are then nearly identical, so there is nothing meaningful to vote over and self consistency collapses back into plain Chain of Thought with no real benefit. When the temperature is too high, the outputs become erratic and incoherent and produce many nonsensical or wrong chains, so the majority vote becomes unreliable. A moderate value such as the 0.7 as an example gave enough variation for genuinely diverse reasoning paths while keeping each path coherent, which is the balance that lets the majority vote work.

In short, temperature trades off diversity against coherence in the sampled reasoning paths, and it must be set to a moderate and non zero value for self consistency to function.

## Task 3: Native Reasoning Model

In this final task, we will load and use a native reasoning model (that is, a System 2 reasoning model) to generate an answer to our question.

In [6]:
native_model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Loading {native_model_name}. This may take a few minutes...")
native_tokenizer = AutoTokenizer.from_pretrained(native_model_name)
native_model = AutoModelForCausalLM.from_pretrained(native_model_name).to(device)
print("DeepSeek-R1 loaded!")

Loading deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B. This may take a few minutes...


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

DeepSeek-R1 loaded!


In [7]:
def run_native_reasoning(question):
    messages = [
        {"role": "system", "content": "Solve the problem. Show your work step-by-step. End your response with 'Answer: <number>'."}, #Instructions for the model to reason step-by-step and provide the final answer
        {"role": "user", "content": question} #Our question is provided as a user message to the model
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) #Apply the chat template to format the messages for the model
    inputs = native_tokenizer(text, return_tensors="pt").to(device) #DeepSeek-R1 prefers simple user prompts without restrictive system instructions

    print("Model is finding the answer (this may take a few minutes)...")
    with torch.no_grad(): #Disable gradient calculation for inference to save memory and computation
        outputs = native_model.generate(**inputs, max_new_tokens=500) #Generate the model's response to the input question, allowing for a maximum of 600 new tokens to be generated.

    full_output = native_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True) #Decode the generated tokens into a string, skipping any special tokens used by the model

    print("\n\n Final answer: \n\n")
    print(full_output) #Print the final answer provided by the model, which is expected to be a concise response to the original question.

run_native_reasoning(question) # Run the question through the DeepSeek-R1 model for native reasoning


Model is finding the answer (this may take a few minutes)...


 Final answer: 


Okay, so Janet's ducks lay 16 eggs per day. That part seems straightforward. Now, she eats three for breakfast every morning. Hmm, so if she has 16 eggs and eats 3 each day for breakfast, that means she uses 3 eggs per day for breakfast. Then, she bakes muffins for her friends every day with four. So, she uses 4 eggs for baking muffins.

Wait, so she uses 3 for breakfast and 4 for muffins. That's a total of 3 + 4 = 7 eggs used daily. So, from her 16 eggs, she uses 7 each day. That leaves her with 16 - 7 = 9 eggs remaining.

She sells the remainder at the farmers' market daily for $2 per fresh duck egg. So, each of those 9 eggs is sold for $2. Therefore, the total she makes is 9 * $2 = $18 per day.

Let me double-check that. 16 eggs total. She eats 3 for breakfast, so 16 - 3 = 13 eggs left. Then, she bakes 4 for friends, so 13 - 4 = 9 eggs left. She sells those 9 for $2 each, so 9 * 2 = $18. Yep, that seems

**Task 3 - Question 1:** What benefit do native reasoning models provide compared with using non-reasoning pre-trained models together with an inference-time technique such as CoT or self-consistency prompting?

TODO - Native reasoning models have the reasoning ability built into the model itself through training, for example by reinforcement learning on long chains of thought or by distillation from a larger reasoning model. Compared with attaching an inference time technique to an ordinary model, this offers several benefits include:

- Built-in reasoning capabilities: These models are specifically trained (often using reinforcement learning) to perform complex multi-step reasoning, making the reasoning process more natural and reliable.

- More robust reasoning: The model has been optimised through training to follow logical chains, reducing common failure modes like hallucinations in intermediate steps; losing track of information across reasoning steps and making arithmetic errors.

- Efficiency: Native reasoning models often require fewer generation attempts to get correct answers compared to self-consistency approaches that need multiple samples.

- Self-verification: Many reasoning models are trained to verify their own reasoning steps, catching errors along the way.

- Better handling of complex problems: Trained on reasoning-heavy tasks, these models develop a deeper understanding of logical structures and mathematical concepts.

- No need for special prompting: They can handle reasoning tasks with simple prompts, whereas non-reasoning models require carefully engineered prompts.

- Transparent reasoning: The model naturally produces interpretable reasoning traces, which can be valuable for understanding and debugging.

**Task 3 - Question 2:** Should we use reasoning models in all cases, that is, for every possible problem? Why or why not?

TODO - No, reasoning models should not be used for all problems because they generate a large number of extra intermediate thinking tokens before answering. This makes them slower, more expensive, and more demanding on energy and compute than a standard model.

For simple tasks such as factual lookups, retrieval, classification, short conversation, and straightforward instruction following, an ordinary model answers correctly and almost instantly, so the extra reasoning is wasted cost and delay and can even be harmful, because the model may overthink a trivial question and talk itself into a wrong answer.

The sensible approach is to match the model to the problem. We should use a native reasoning model when the task genuinely requires several steps of reasoning, such as hard mathematics, logic, planning, complex coding, or proofs, and use a cheaper ordinary model when the task is simple or when speed matters. Fundamentally it is a trade off between accuracy on one side and cost, speed, and energy on the other.

